# Ensemble weight search and justification

Replaces the ad-hoc three-combination trial with a systematic, leakage-free
determination of the DeepKriging / CNN / LightGBM blend, and quantifies how much
the choice actually matters.

**What this notebook shows**
1. **Full simplex grid search** over all convex weights (w_DK + w_CNN + w_LGBM = 1,
   each >= 0) on the **OOF** predictions -> the empirical optimum and the response
   surface.
2. **Constrained optimisation (stacking)** - non-negative weights summing to 1 fit
   to minimise OOF error, i.e. the data-driven optimum without a discrete grid.
3. **Candidate comparison** - manual (0.40/0.35/0.25), equal (1/3 each),
   grid-optimum, and stacking-optimum, on both OOF and the untouched test set.
4. **Robustness** - flatness of the surface around the optimum and per-fold
   stability of the optimal weights.
5. **Significance** - paired station-block bootstrap: is the optimum *significantly*
   better than the manual choice, or statistically indistinguishable?

**Why fitting weights on OOF is valid:** the base-model OOF predictions are already
leakage-free (each station held out once). The blend has only two free parameters,
so overfitting risk is negligible; the independent 16-station test set then confirms
the choice.

Because the observation is fixed, minimising the error of the *corrected PM2.5*
(`PM25 - w.bias`) is identical to fitting the weighted **bias** prediction to the
`True_Bias`, which is what the optimiser below uses.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

RESULTS = r"results"
FOLDER = RESULTS + r"\Ensemble_DK_CNN_LGBM_403525"
OOF_FILE = FOLDER + r"\Ensemble_OOF_predictions.csv"
TEST_FILE = FOLDER + r"\Ensemble_final_test_predictions.csv"

OBS, MUSICA = "AURN_Observation", "PM25"
BIAS_COLS = ["DeepKriging_Predicted_Bias", "CNN_Predicted_Bias", "LightGBM_Predicted_Bias"]
NAMES = ["DeepKriging", "CNN", "LightGBM"]
MANUAL_W = np.array([0.40, 0.35, 0.25])   # weights currently used in the paper

oof = pd.read_csv(OOF_FILE)
oof["station"] = oof["station"].astype(str).str.strip()
test = pd.read_csv(TEST_FILE)
test["station"] = test["station"].astype(str).str.strip()
print(f"OOF: {len(oof):,} rows / {oof['station'].nunique()} stations")
print(f"Test: {len(test):,} rows / {test['station'].nunique()} stations")
oof[BIAS_COLS + [OBS, MUSICA]].describe().round(3)

OOF: 465,659 rows / 63 stations
Test: 127,695 rows / 16 stations


,DeepKriging_Predicted_Bias,CNN_Predicted_Bias,LightGBM_Predicted_Bias,AURN_Observation,PM25
count,465659.000,465659.000,465659.000,465659.000,465659.0
mean,-9.616,-9.480,-10.018,10.117,0.0
std,7.321,6.699,6.227,8.621,0.0
min,-186.650,-138.390,-103.479,0.000,0.0
25%,-11.544,-11.299,-12.109,4.800,0.0
50%,-7.369,-7.327,-8.257,7.700,0.0
75%,-5.065,-5.192,-5.988,12.700,0.0
max,2.815,-1.482,0.121,337.100,0.0


In [2]:
# Pre-extract arrays once (fast repeated evaluation over many weight vectors).
B_oof = oof[BIAS_COLS].to_numpy(float)      # (n, 3) component bias predictions
y_oof = oof[OBS].to_numpy(float)
m_oof = oof[MUSICA].to_numpy(float)
B_test = test[BIAS_COLS].to_numpy(float)
y_test = test[OBS].to_numpy(float)
m_test = test[MUSICA].to_numpy(float)

def metrics(w, B, m, y):
    """Weighted-blend corrected PM2.5 metrics vs AURN for weight vector w."""
    corrected = m - B @ w
    err = corrected - y
    ss_res = (err ** 2).sum()
    ss_tot = ((y - y.mean()) ** 2).sum()
    return {
        "R2": 1 - ss_res / ss_tot,
        "MAE": np.abs(err).mean(),
        "RMSE": np.sqrt((err ** 2).mean()),
        "BIAS": err.mean(),
    }

print("Manual weights 0.40/0.35/0.25 on OOF:", {k: round(v, 4) for k, v in metrics(MANUAL_W, B_oof, m_oof, y_oof).items()})

Manual weights 0.40/0.35/0.25 on OOF: {'R2': np.float64(0.6487), 'MAE': np.float64(3.3038), 'RMSE': np.float64(5.1099), 'BIAS': np.float64(-0.4479)}


In [3]:
# ---- 1. Full simplex grid search on OOF (step = 0.05) ----
STEP = 0.05
grid = []
levels = np.round(np.arange(0, 1 + 1e-9, STEP), 4)
for w_dk in levels:
    for w_cnn in levels:
        w_lgbm = round(1.0 - w_dk - w_cnn, 4)
        if w_lgbm < -1e-9 or w_lgbm > 1 + 1e-9:
            continue
        w = np.array([w_dk, w_cnn, w_lgbm])
        r = metrics(w, B_oof, m_oof, y_oof)
        grid.append({"w_DK": w_dk, "w_CNN": w_cnn, "w_LGBM": w_lgbm, **r})
grid = pd.DataFrame(grid)
print(f"Evaluated {len(grid)} weight combinations on the simplex (step {STEP}).")

opt_mae = grid.loc[grid["MAE"].idxmin()]
opt_rmse = grid.loc[grid["RMSE"].idxmin()]
print("\nGrid optimum (min OOF MAE): ",
      opt_mae[["w_DK", "w_CNN", "w_LGBM"]].to_dict(), "-> MAE", round(opt_mae["MAE"], 4))
print("Grid optimum (min OOF RMSE):",
      opt_rmse[["w_DK", "w_CNN", "w_LGBM"]].to_dict(), "-> RMSE", round(opt_rmse["RMSE"], 4))
print("\nTop 8 by OOF MAE:")
grid.sort_values("MAE").head(8).round(4)

Evaluated 231 weight combinations on the simplex (step 0.05).

Grid optimum (min OOF MAE):  {'w_DK': 0.5, 'w_CNN': 0.35, 'w_LGBM': 0.15} -> MAE 3.2954
Grid optimum (min OOF RMSE): {'w_DK': 0.5, 'w_CNN': 0.35, 'w_LGBM': 0.15} -> RMSE 5.0948

Top 8 by OOF MAE:


,w_DK,w_CNN,w_LGBM,R2,MAE,RMSE,BIAS
172,0.50,0.35,0.15,0.6508,3.2954,5.0948,-0.4881
161,0.45,0.40,0.15,0.6502,3.2961,5.0987,-0.4950
173,0.50,0.40,0.10,0.6505,3.2963,5.0966,-0.5151
160,0.45,0.35,0.20,0.6502,3.2974,5.0993,-0.4680
171,0.50,0.30,0.20,0.6505,3.2980,5.0971,-0.4612
162,0.45,0.45,0.10,0.6497,3.2982,5.1024,-0.5219
183,0.55,0.35,0.10,0.6506,3.2982,5.0964,-0.5082
182,0.55,0.30,0.15,0.6506,3.2986,5.0964,-0.4813


In [4]:
# ---- 2. Constrained optimisation (stacking): non-negative weights summing to 1 ----
def objective(w, loss):
    r = metrics(w, B_oof, m_oof, y_oof)
    return r[loss]

cons = ({"type": "eq", "fun": lambda w: w.sum() - 1.0},)
bounds = [(0, 1)] * 3
x0 = np.array([1 / 3, 1 / 3, 1 / 3])

stack_mae = minimize(objective, x0, args=("MAE",), method="SLSQP", bounds=bounds, constraints=cons)
stack_rmse = minimize(objective, x0, args=("RMSE",), method="SLSQP", bounds=bounds, constraints=cons)
w_stack_mae = stack_mae.x
w_stack_rmse = stack_rmse.x
print("Stacking optimum (min OOF MAE): ", np.round(w_stack_mae, 4), "-> MAE", round(stack_mae.fun, 4))
print("Stacking optimum (min OOF RMSE):", np.round(w_stack_rmse, 4), "-> RMSE", round(stack_rmse.fun, 4))

Stacking optimum (min OOF MAE):  [0.4862 0.3702 0.1436] -> MAE 3.2952
Stacking optimum (min OOF RMSE): [0.5135 0.3449 0.1417] -> RMSE 5.0946


In [5]:
# ---- 3. Candidate comparison on OOF and independent test ----
candidates = {
    "Manual (0.40/0.35/0.25)": MANUAL_W,
    "Equal (1/3 each)": np.array([1 / 3, 1 / 3, 1 / 3]),
    "Grid-opt (MAE)": opt_mae[["w_DK", "w_CNN", "w_LGBM"]].to_numpy(float),
    "Grid-opt (RMSE)": opt_rmse[["w_DK", "w_CNN", "w_LGBM"]].to_numpy(float),
    "Stacking-opt (MAE)": w_stack_mae,
    "Stacking-opt (RMSE)": w_stack_rmse,
}

rows = []
for name, w in candidates.items():
    o = metrics(w, B_oof, m_oof, y_oof)
    t = metrics(w, B_test, m_test, y_test)
    rows.append({
        "weighting": name,
        "w_DK": round(w[0], 3), "w_CNN": round(w[1], 3), "w_LGBM": round(w[2], 3),
        "OOF_R2": round(o["R2"], 4), "OOF_MAE": round(o["MAE"], 4), "OOF_RMSE": round(o["RMSE"], 4),
        "Test_R2": round(t["R2"], 4), "Test_MAE": round(t["MAE"], 4), "Test_RMSE": round(t["RMSE"], 4),
    })
comparison = pd.DataFrame(rows)
comparison.to_csv(FOLDER + r"\Ensemble_weight_search_comparison.csv", index=False)
comparison

,weighting,w_DK,w_CNN,w_LGBM,OOF_R2,OOF_MAE,OOF_RMSE,Test_R2,Test_MAE,Test_RMSE
0,Manual (0.40/0.35/0.25),0.400,0.350,0.250,0.6487,3.3038,5.1099,0.6165,3.2019,5.0949
1,Equal (1/3 each),0.333,0.333,0.333,0.6451,3.3222,5.1364,0.6127,3.2205,5.1201
2,Grid-opt (MAE),0.500,0.350,0.150,0.6508,3.2954,5.0948,0.6194,3.1900,5.0756
3,Grid-opt (RMSE),0.500,0.350,0.150,0.6508,3.2954,5.0948,0.6194,3.1900,5.0756
4,Stacking-opt (MAE),0.486,0.370,0.144,0.6507,3.2952,5.0954,0.6187,3.1923,5.0806
5,Stacking-opt (RMSE),0.513,0.345,0.142,0.6508,3.2958,5.0946,0.6197,3.1893,5.0737


In [6]:
# ---- 4a. Robustness: flatness of the response surface around the manual choice ----
best_mae = grid["MAE"].min()
within = grid[grid["MAE"] <= best_mae + 0.02]   # within 0.02 ug/m3 of the optimum
print(f"Best OOF MAE = {best_mae:.4f}")
print(f"Manual OOF MAE = {metrics(MANUAL_W, B_oof, m_oof, y_oof)['MAE']:.4f} "
      f"(gap to optimum = {metrics(MANUAL_W, B_oof, m_oof, y_oof)['MAE'] - best_mae:.4f} ug/m3)")
print(f"{len(within)}/{len(grid)} grid points lie within 0.02 ug/m3 of the optimum "
      f"-> the surface is {'FLAT' if len(within) > 20 else 'peaked'}")
print("\nRange of near-optimal weights (within 0.02 of best MAE):")
print(within[["w_DK", "w_CNN", "w_LGBM"]].describe().loc[["min", "max"]].round(3))

Best OOF MAE = 3.2954
Manual OOF MAE = 3.3038 (gap to optimum = 0.0084 ug/m3)
38/231 grid points lie within 0.02 ug/m3 of the optimum -> the surface is FLAT

Range of near-optimal weights (within 0.02 of best MAE):
     w_DK  w_CNN  w_LGBM
min  0.35   0.20     0.0
max  0.65   0.55     0.3


In [7]:
# ---- 4b. Robustness: per-fold optimal weights (are they stable across folds?) ----
fold_opt = []
for fid, g in oof.groupby("Fold_ID"):
    Bf = g[BIAS_COLS].to_numpy(float); mf = g[MUSICA].to_numpy(float); yf = g[OBS].to_numpy(float)
    res = minimize(lambda w: np.abs((mf - Bf @ w) - yf).mean(), x0,
                   method="SLSQP", bounds=bounds, constraints=cons)
    fold_opt.append({"Fold_ID": int(fid), "w_DK": res.x[0], "w_CNN": res.x[1], "w_LGBM": res.x[2]})
fold_opt = pd.DataFrame(fold_opt)
print("Per-fold stacking-optimal weights (stability check):")
print(fold_opt.round(3).to_string(index=False))
print("\nStd across folds:", fold_opt[["w_DK", "w_CNN", "w_LGBM"]].std().round(3).to_dict())

Per-fold stacking-optimal weights (stability check):
 Fold_ID  w_DK  w_CNN  w_LGBM
       1 0.347  0.581   0.072
       2 0.518  0.346   0.135
       3 0.493  0.323   0.184
       4 0.565  0.318   0.117
       5 0.548  0.256   0.196

Std across folds: {'w_DK': 0.087, 'w_CNN': 0.125, 'w_LGBM': 0.051}
